In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel
from PIL import Image

model_name = "khhuang/chart-to-table"
model = VisionEncoderDecoderModel.from_pretrained(model_name).cuda()
processor = DonutProcessor.from_pretrained(model_name)

image_path = "../../type2/simple/air_humidity_evaporation_rate_0.png"

input_prompt = "<data_table_generation> <s_answer>"

img = Image.open(image_path)
pixel_values = processor(img.convert("RGB"), random_padding=False, return_tensors="pt").pixel_values
pixel_values = pixel_values.cuda()
decoder_input_ids = processor.tokenizer(input_prompt, add_special_tokens=False, return_tensors="pt", max_length=510).input_ids.cuda()#.squeeze(0)


outputs = model.generate(
        pixel_values.cuda(),
        decoder_input_ids=decoder_input_ids.cuda(),
        max_length=model.decoder.config.max_position_embeddings,
        early_stopping=True,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
        use_cache=True,
        num_beams=4,
        bad_words_ids=[[processor.tokenizer.unk_token_id]],
        return_dict_in_generate=True,
    )
    


In [2]:
sequence = processor.batch_decode(outputs.sequences)[0]
sequence = sequence.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
extracted_table = sequence.split("<s_answer>")[1].strip()

In [ ]:
print('\n'.join(extracted_table.split('&&&')))

In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel
from PIL import Image
import os
import json
import torch
from tqdm import tqdm

device = 'cuda:1' if torch.cuda.is_available() else 'cpu'
model_name = "c2t"

model = VisionEncoderDecoderModel.from_pretrained("khhuang/chart-to-table").cuda()
processor = DonutProcessor.from_pretrained("khhuang/chart-to-table")

types = ['t1', 't2', 't3']
directories = ['../../type1/simple', '../../type2/simple', '../../type3/simple']

os.makedirs('../../model_responses/chart_to_table/c2t', exist_ok=True)

for i in range(len(types)):
    chart_type = types[i]
    directory = directories[i]
    
    all_charts = json.load(open(f'{chart_type}_charts.json'))    
    
    # os.makedirs(f'../../model_responses/chart_to_table/{model_name}/{chart_type}', exist_ok=True)
    
    for key, value in tqdm(all_charts.items()):
        chart_name = key
        charts = value
        
        images = [Image.open(f'{directory}/{chart}').convert("RGB") for chart in charts]
        pixel_values = processor(images, random_padding=False, return_tensors="pt").pixel_values
        input_prompts = ["<data_table_generation> <s_answer>"] * len(images)
        decoder_input_ids = processor.tokenizer(input_prompts, add_special_tokens=False, return_tensors="pt", max_length=510).input_ids.cuda().squeeze(0)
        
        outputs = model.generate(
            pixel_values.cuda(),
            decoder_input_ids=decoder_input_ids.cuda(),
            max_length=model.decoder.config.max_position_embeddings,
            early_stopping=True,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True,
            num_beams=4,
            bad_words_ids=[[processor.tokenizer.unk_token_id]],
            return_dict_in_generate=True,
        )

        sequences = processor.batch_decode(outputs.sequences)
        sequences = [sequence.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "") for sequence in sequences] 
        tables = [sequence.split("<s_answer>")[1].strip() for sequence in sequences]
        print("table")
        print(tables)
        # with open(f'../../model_responses/chart_to_table/{model_name}/{chart_type}/{chart_name}.json', 'w') as f:
        #     json.dump(tables, f, indent=4)

        del images, pixel_values, input_prompts, decoder_input_ids, outputs, sequences, tables 